In [1]:
import sys, json
sys.path.append("..")
%load_ext autoreload
%autoreload 2

from src.tokenizer import CharTokenizer

def load(name):
    return [json.loads(line) for line in open(f"../data/raw/{name}.jsonl", encoding="utf-8")]

train, val, test = load("train"), load("val"), load("test")

texts = [r["question"] for r in train] + [r["sql"] for r in train]
tok = CharTokenizer.build(texts)
print("vocab size:", tok.vocab_size)
print(tok.tokens)

vocab size: 73
['<pad>', '<unk>', '<sep>', '<eos>', ' ', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'W', 'Y', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z']


In [2]:
# 1. Round trip: encoding then decoding must give back the same text
sql = train[0]["sql"]
assert tok.decode(tok.encode(sql)) == sql
print("round trip OK")

# 2. One full example, and where the SQL starts
q, s = train[0]["question"], train[0]["sql"]
ids, sql_start = tok.encode_example(q, s)
print(ids[:12], "...")
print("length:", len(ids), "| sql starts at:", sql_start)
print("token at sql_start:", tok.itos[ids[sql_start]], "(should be the first SQL character)")
print("first and last special ids:", ids[sql_start - 1] == tok.sep_id, ids[-1] == tok.eos_id)

# 3. No unknown characters anywhere in val or test
for row in val + test:
    assert tok.unk_id not in tok.encode(row["question"] + row["sql"])
print("no unknown characters in val/test")

# 4. How long is our longest example?
longest = max(len(r["question"]) + len(r["sql"]) + 2 for r in train + val + test)
print("longest example:", longest, "tokens")

round trip OK
[48, 69, 52, 65, 48, 54, 52, 4, 63, 65, 56, 50] ...
length: 105 | sql starts at: 42
token at sql_start: S (should be the first SQL character)
first and last special ids: True True
no unknown characters in val/test
longest example: 265 tokens
